<a href="https://colab.research.google.com/github/nawroz-m/ML_learning/blob/supervised/Assignment01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Compare 3 different classifiers in a robust way on the 4 datasets provided.

Use the Frideman test performing a 5-time 2-fold cross validation for each classifier on each dataset.

Include the code and the report in a .zip archive and upload it.

In [50]:
from google.colab import drive
import scipy.io as sci
from sklearn.svm import SVC
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from scipy.stats import friedmanchisquare

In [3]:
# Connect to the drive
drive.flush_and_unmount()
drive.mount('/content/drive')

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive


In [4]:
# dataset path
dataset_path = "/content/drive/MyDrive/Supervised/Datasets"

In [5]:
# get the first dataset
ds = sci.loadmat(f'{dataset_path}/dataset1.mat')
X_train_ds, X_test_ds, y_train_ds, y_test_ds = train_test_split(ds['data'], ds['labels'], test_size=0.2)
y_test_ds[:3]

array([[2],
       [1],
       [1]], dtype=uint8)

In [57]:
# train classifier in robust way
def train_robust_classifier(dataset=None, models=None, iteration=1):
  # prepare the dataset for training and testing
  accuracy = {name: [] for name in models.keys()}
  for ntimes in range(iteration):
    X_train, X_test, y_train, y_test = train_test_split(dataset['data'],
                                                        dataset['labels'],
                                                        test_size=0.5,
                                                        shuffle=True)
    # train in robust model way
    for name, model in models.items():
      model.fit(X_train, y_train.ravel())
      # predict on test dataset
      y_pred_test = model.predict(X_test)
      # get accuracy on test dataset
      accuracy[name].append(np.mean(y_pred_test == y_test.squeeze()))
      # reverse the training and testing proccess
      model.fit(X_test, y_test.ravel())
      y_pred_train = model.predict(X_train)
      accuracy[name].append(np.mean(y_pred_train == y_train.squeeze()))

  return accuracy

In [40]:
# Read and train multiple classifier on 4 different dataset
seed = 42
iteration = 5
def classifier_model(path=None, models=None):
  accuracy5x2 = []
  for i in range(1, 5):
    # Read the dataset
    dataset = sci.loadmat(file_name=f'{path}/dataset{i}.mat')
    # Train the model on the training dataset
    accuracy = train_robust_classifier(dataset, models, iteration)
    accuracy5x2.append(accuracy)
  return accuracy5x2

In [58]:
models = {
    'svm_lin': SVC(C=1.0, kernel='linear'),
    # 'svm_rbf': SVC(C=1.0, kernel='rbf'),
    'tree': DecisionTreeClassifier(criterion='gini', max_depth=20),
    'knn': KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2)
    }
accuracy5x2 = classifier_model(dataset_path, models=models)

In [79]:
# Create a summary table for each classifier on each dataset.
summary_table = []
headers = ['dataset', 'name', 'mean', 'std']
for i, accuracy in enumerate(accuracy5x2):
  for name, acc in accuracy.items():
    summary_table.append([f'D{i+1}', name, np.mean(acc), np.std(acc)])
summary_table = pd.DataFrame(summary_table, columns=headers)
summary_table

,dataset,name,mean,std
0,D1,svm_lin,1.000000,0.000000
1,D1,tree,0.994667,0.002667
2,D1,knn,1.000000,0.000000
3,D2,svm_lin,0.879333,0.018726
4,D2,tree,0.802000,0.029672
5,D2,knn,0.856667,0.019149
6,D3,svm_lin,0.666667,0.022608
7,D3,tree,0.888667,0.018690
8,D3,knn,0.914667,0.011372
9,D4,svm_lin,0.556000,0.040382


In [90]:
# Create comparison matrix
comparison_table = []
headers = ['Dataset', 'svm_lin', 'tree', 'knn']
for i, accuracy in enumerate(accuracy5x2):
  svm_lin_mean = np.mean(accuracy[headers[1]])
  tree_mean = np.mean(accuracy[headers[2]])
  knn_mean = np.mean(accuracy[headers[3]])

  comparison_table.append([f'D{i+1}', svm_lin_mean, tree_mean, knn_mean])
comparison_table = pd.DataFrame(comparison_table, columns=headers)
comparison_table

,Dataset,svm_lin,tree,knn
0,D1,1.000000,0.994667,1.000000
1,D2,0.879333,0.802000,0.856667
2,D3,0.666667,0.888667,0.914667
3,D4,0.556000,0.961000,0.964667


In [89]:
# Create Average Rank table

In [86]:
# Create Friedman test
stat, p = friedmanchisquare(comparison_table['svm_lin'], comparison_table['tree'], comparison_table['knn'])

print("Friedman statistic:", stat)
print("p-value:", p)

Friedman statistic: 2.8
p-value: 0.24659696394160646



**Meaning of Friedman statistic**
```
p < 0.05 ––>	classifiers perform significantly differently

p ≥ 0.05 ––>	no significant difference

```


### Final Report

The Friedman test was applied to compare the performance of the three classifiers
(SVM, Decision Tree, and KNN) across four datasets using the results obtained
from 5×2 cross-validation.

The test produced:

```
    χ² = 2.8
    p = 0.2466
```


Since the p-value is greater than the significance level α = 0.05,
we fail to reject the null hypothesis. Therefore, there is no
statistically significant difference between the classifiers'
performance across the datasets.